# Query-budget reduction: Table 4 data

Collects the `budget_summary.csv` written by the 16 `*-budgets.ipynb` notebooks
(white/black-box × frequency/topic × cache/no-cache × PG19/PubMed) and builds
two Table 4-style tables, one per template rule: `random` (each noun keeps its own random
templates) and `shared` (every noun keeps the same template indices).

Gain over random = (Macro-F1 − 1/K) / (1/K), with K = 6 classes for frequency
and K = 5 topics.

**Topic columns use the final experiments** (`white-box-cat-final`, `black-box-cat-final`): the revised method
with 500 trees per ensemble, matching the frequency experiments. One prediction
per target noun from system signals only (no query text or query length), drift-corrected, evaluated with
nested noun cross-validation. The original topic notebooks could read the topic from the question templates.

Outputs in `budget_reduction_tables_final/`:
- `budget_results_long.csv`: every run, one row each.
- `table4_random_templates.csv/.tex` and `table4_shared_templates.csv/.tex`: one table per
  template rule (the LaTeX needs `booktabs`, `longtable`, `multirow` and `xcolor` with the
  `table` option).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(".")
OUTPUT_DIRECTORY = Path("budget_reduction_tables_final")
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = (
    # (label, folder, task, box)
    ("White-box. Content frequency", "white-box-freq", "freq", "white"),
    ("Black-box. Content frequency", "black-box-freq", "freq", "black"),
    # Topic results come from the final experiments (revised leak-free method,
    # 500 trees); see classificators/final_topic.py.
    ("White-box. Topic of use", "white-box-cat-final", "topic", "white"),
    ("Black-box. Topic of use", "black-box-cat-final", "topic", "black"),
)
DATASETS = ("pg19", "pubmed")
DATASET_LABELS = {"pg19": "PG19", "pubmed": "PubMed"}
CACHE_SETTINGS = {"no_cache": False, "cache": True}
CACHE_LABELS = {False: "No cache", True: "Cache"}
MODELS = ("smol", "qwen", "granite")
TEMPLATE_RULES = ("random", "shared")
TEMPLATE_LABELS = {"random": "Random templates", "shared": "Shared templates"}
BUDGETS_PCT = (100, 75, 50, 25, 10, 5)

## 1. Collect every budget summary

In [2]:
frames = []
missing = []
for label, folder, task, box in EXPERIMENTS:
    for cache in CACHE_SETTINGS.values():
        variant = "cache" if cache else "no-cache"
        for dataset in DATASETS:
            summary_path = (
                ROOT
                / folder
                / variant
                / f"{dataset}_{'cache_' if cache else ''}{box}_box_{task}_budget_results"
                / "budget_summary.csv"
            )
            if summary_path.exists():
                frames.append(pd.read_csv(summary_path))
            else:
                missing.append(str(summary_path))

if missing:
    print("Missing summaries (run those notebooks first):")
    print("\n".join(missing))

results = pd.concat(frames, ignore_index=True)
results.to_csv(OUTPUT_DIRECTORY / "budget_results_long.csv", index=False)
print(f"{len(results)} runs collected")

576 runs collected


## 2. Tables per template rule

Rows: cache setting → dataset → model → retained queries. Columns follow Table 4.

In [3]:
def build_table() -> pd.DataFrame:
    blocks = []
    for label, _folder, task, _box in EXPERIMENTS:
        block = (
            results.loc[results["experiment"] == label]
            .assign(
                cache=lambda frame: frame["cache"].map(CACHE_LABELS),
                template_rule=lambda frame: frame["template_rule"].map(TEMPLATE_LABELS),
            )
            .set_index(["template_rule", "cache", "dataset", "model", "budget_pct"])
            [["macro_f1", "gain_over_random_pct", "ow_accuracy"]]
            .rename(
                columns={
                    "macro_f1": "Macro-F1",
                    "gain_over_random_pct": "Macro-F1 gain over random (%)",
                    "ow_accuracy": "OWAcc",
                }
            )
        )
        if task == "topic":
            block = block.drop(columns="OWAcc")
        block.columns = pd.MultiIndex.from_product([[label], block.columns])
        blocks.append(block)
    table = pd.concat(blocks, axis=1)
    order = pd.MultiIndex.from_product(
        [
            [TEMPLATE_LABELS[rule] for rule in TEMPLATE_RULES],
            [CACHE_LABELS[c] for c in CACHE_SETTINGS.values()],
            [DATASET_LABELS[d] for d in DATASETS],
            MODELS,
            BUDGETS_PCT,
        ],
        names=["Templates", "Cache", "D", "Model", "Retained queries (%)"],
    )
    return table.reindex(order)


GREEN = "1F5C2E"
RED = "E8A0A0"
MAX_GAIN = 350.0


def gain_cell(value: float) -> str:
    if pd.isna(value):
        return "--"
    text = f"{value:.0f}\\%"
    if value < 0:
        return f"\\cellcolor[HTML]{{{RED}}}{text}"
    strength = int(round(min(value, MAX_GAIN) / MAX_GAIN * 100))
    cell = f"\\cellcolor{{tablegreen!{strength}}}"
    return cell + (f"\\textcolor{{white}}{{{text}}}" if strength > 60 else text)


def to_latex(table: pd.DataFrame, caption: str) -> str:
    """Render as a longtable, since 72 rows do not fit on one page."""
    columns = list(table.columns)
    n_columns = 2 + len(columns)
    header_groups = []
    for label, _folder, task, _box in EXPERIMENTS:
        width = 3 if task == "freq" else 2
        header_groups.append(f"\\multicolumn{{{width}}}{{c}}{{{label}}}")
    sub_header = [
        {
            "Macro-F1": "Macro-F$_1$",
            "Macro-F1 gain over random (%)": "Gain over random",
            "OWAcc": "OWAcc",
        }[sub]
        for _label, sub in columns
    ]
    header = [
        "\\toprule",
        " & & " + " & ".join(header_groups) + " \\\\",
        "Model & $b$ & " + " & ".join(sub_header) + " \\\\",
        "\\midrule",
    ]
    lines = [
        "\\definecolor{tablegreen}{HTML}{" + GREEN + "}",
        "\\begingroup\\footnotesize",
        "\\setlength{\\tabcolsep}{3pt}",
        "\\begin{longtable}{ll" + "c" * len(columns) + "}",
        f"\\caption{{{caption}}}\\\\",
        *header,
        "\\endfirsthead",
        *header,
        "\\endhead",
        "\\bottomrule",
        "\\endlastfoot",
    ]
    sections = dict.fromkeys(index[:3] for index in table.index)
    for rule_label, cache_label, dataset in sections:
        lines.append("\\midrule")
        lines.append(
            f"\\multicolumn{{{n_columns}}}{{l}}"
            f"{{\\textbf{{{rule_label} -- {cache_label} -- {dataset}}}}} \\\\"
        )
        for model in MODELS:
            block = table.loc[(rule_label, cache_label, dataset, model)]
            for row_number, (budget, row) in enumerate(block.iterrows()):
                model_cell = (
                    f"\\multirow{{{len(block)}}}{{*}}{{{model}}}" if row_number == 0 else ""
                )
                cells = []
                for (_label, sub), value in row.items():
                    if sub.startswith("Macro-F1 gain"):
                        cells.append(gain_cell(value))
                    else:
                        cells.append("--" if pd.isna(value) else f"{value:.2f}")
                lines.append(
                    f"{model_cell} & {budget}\\% & " + " & ".join(cells) + " \\\\"
                )
            if model != MODELS[-1]:
                lines.append(f"\\cmidrule(l){{1-{n_columns}}}")
    lines += ["\\end{longtable}", "\\endgroup"]
    return "\n".join(lines)


table = build_table()
for rule in TEMPLATE_RULES:
    rule_table = table.loc[TEMPLATE_LABELS[rule]]
    rule_table.to_csv(OUTPUT_DIRECTORY / f"table4_{rule}_templates.csv")
    (OUTPUT_DIRECTORY / f"table4_{rule}_templates.tex").write_text(
        to_latex(
            pd.concat({TEMPLATE_LABELS[rule]: rule_table}, names=["Templates"]),
            f"Attack performance under query budget reductions ({rule} templates)",
        ),
        encoding="utf-8",
    )
print(f"Tables written to: {OUTPUT_DIRECTORY.resolve()}")

Tables written to: /Users/marinabuitrago/Desktop/experiments-final/classificators/budget_reduction_tables_final


## 3. View the tables

One table per template rule (72 rows each), with the gain column coloured as in the
LaTeX version (green for gains, red below random).

In [4]:
gain_columns = [column for column in table.columns if column[1].startswith("Macro-F1 gain")]
value_columns = [column for column in table.columns if column not in gain_columns]
GREEN_RGB = tuple(int(GREEN[i : i + 2], 16) for i in (0, 2, 4))


def gain_style(value: float) -> str:
    if pd.isna(value):
        return ""
    if value < 0:
        return f"background-color: #{RED}; color: #0b0b0b"
    strength = min(value, MAX_GAIN) / MAX_GAIN
    red, green, blue = (round(255 + (channel - 255) * strength) for channel in GREEN_RGB)
    text = "white" if strength > 0.6 else "#0b0b0b"
    return f"background-color: rgb({red}, {green}, {blue}); color: {text}"


def style_table(frame: pd.DataFrame, caption: str):
    return (
        frame.style.format("{:.2f}", subset=value_columns, na_rep="--")
        .format("{:.0f}%", subset=gain_columns, na_rep="--")
        .map(gain_style, subset=gain_columns)
        .set_caption(caption)
        .set_properties(**{"text-align": "center"})
        .set_table_styles(
            [
                {"selector": "th", "props": [("text-align", "center")]},
                {"selector": "caption", "props": [("font-size", "1.3em"), ("font-weight", "bold")]},
            ]
        )
    )


from IPython.display import display

for rule in TEMPLATE_RULES:
    display(style_table(table.loc[TEMPLATE_LABELS[rule]], TEMPLATE_LABELS[rule]))